# Pipeline Validation

Validates that both venue clients can fetch live orderbooks. Picks one well-known market on each venue and prints the orderbook side-by-side.

This is the only Phase 1 deliverable. No analysis yet.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../src")

from pm_micro.clients import kalshi, polymarket

## Kalshi

Fetch a known active market. Using a Bitcoin price market as a likely-stable choice.

In [2]:
# Find a live Kalshi market — try a Bitcoin price one
# If this ticker is expired by the time you run this, replace with any active market ticker
# from https://kalshi.com/markets
KALSHI_TICKER = "FEDHIKE-26DEC31"

try:
    market = kalshi.get_market(KALSHI_TICKER)
    print("Market:", market.get("market", {}).get("title", "unknown"))
    orderbook = kalshi.get_orderbook(KALSHI_TICKER)
    print("\nOrderbook (raw):")
    print(orderbook)
except Exception as e:
    print(f"Kalshi fetch failed: {e}")
    print("Try replacing KALSHI_TICKER above with an active ticker from kalshi.com")

Market: Will the Federal Reserve hike rates by December 31, 2026?

Orderbook (raw):
{'orderbook_fp': {'no_dollars': [['0.0100', '56563.20'], ['0.0200', '10003.00'], ['0.0500', '1879.00'], ['0.0600', '6666.00'], ['0.1000', '210.00'], ['0.1100', '11.00'], ['0.1300', '888.00'], ['0.1400', '128.63'], ['0.1800', '24.65'], ['0.2000', '29.24'], ['0.2100', '4991.00'], ['0.2200', '5443.38'], ['0.2300', '400.00'], ['0.2400', '83.34'], ['0.2500', '2074.98'], ['0.2600', '387.00'], ['0.2700', '195.00'], ['0.2800', '100.00'], ['0.3100', '206.12'], ['0.3300', '250.00'], ['0.3400', '150.00'], ['0.3500', '230.00'], ['0.3800', '5.00'], ['0.3900', '50.00'], ['0.4500', '80.00'], ['0.4800', '10.00'], ['0.4900', '50.00'], ['0.5100', '10.00'], ['0.5200', '10.00'], ['0.5300', '50.00'], ['0.5400', '535.97'], ['0.5500', '1674.11'], ['0.5600', '200.00'], ['0.5700', '44.00'], ['0.5800', '1658.00'], ['0.5900', '5.00']], 'yes_dollars': [['0.0100', '3241621.00'], ['0.0200', '32259.00'], ['0.0300', '28868.00'], ['0.0

## Polymarket

Find a market via Gamma API, then fetch its orderbook via the CLOB.

In [3]:
# Search for a live market
results = polymarket.search_markets("bitcoin", limit=20)
print(f"Found {len(results)} markets matching 'bitcoin'")
for r in results[:5]:
    print(f"  - {r['question'][:80]}  (vol: {r.get('volume', 'n/a')})")

Found 1 markets matching 'bitcoin'
  - Will bitcoin hit $1m before GTA VI?  (vol: 4292523.107865519)


In [4]:
# Pick the first result with valid clobTokenIds and fetch its orderbook
if results:
    market = results[0]
    print(f"Using market: {market['question']}")
    token_ids = market.get("clobTokenIds")
    if isinstance(token_ids, str):
        import json as _json
        token_ids = _json.loads(token_ids)
    print(f"Token IDs: {token_ids}")

    if token_ids and len(token_ids) >= 1:
        yes_token = token_ids[0]
        book = polymarket.get_orderbook(yes_token)
        print(f"\nYES token orderbook:")
        print(f"  Bids: {book.bids[:5] if book.bids else 'none'}")
        print(f"  Asks: {book.asks[:5] if book.asks else 'none'}")
        mid = polymarket.get_midpoint(yes_token)
        print(f"  Midpoint: {mid}")
else:
    print("No markets found — try a different query in cell above")

Using market: Will bitcoin hit $1m before GTA VI?
Token IDs: ['105267568073659068217311993901927962476298440625043565106676088842803600775810', '91863162118308663069733924043159186005106558783397508844234610341221325526200']

YES token orderbook:
  Bids: [OrderSummary(price='0.001', size='424334'), OrderSummary(price='0.002', size='147500'), OrderSummary(price='0.003', size='10000'), OrderSummary(price='0.01', size='5'), OrderSummary(price='0.02', size='6125')]
  Asks: [OrderSummary(price='0.999', size='17500'), OrderSummary(price='0.997', size='10000'), OrderSummary(price='0.99', size='70510'), OrderSummary(price='0.989', size='10000'), OrderSummary(price='0.979', size='800')]
  Midpoint: 0.4925


## Side-by-side summary

Both clients work. Pipeline is validated. Next phase: market mapping.